In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
import ehtim as eh

from torchvision.transforms import v2
import data.dataset_img as ds
import data.CI_torch_v2 as CI
import torchvision
import models.model_DIReCT as mlmodel
import os
from tqdm.auto import tqdm
from models.CLloss import SupConLoss
import data.imgTransforms as imgTransforms

import glob

from xlogger import *

import os

# use gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tforms = imgTransforms.imgTransforms()

model_name = 'DIReCT_v2'
append = True
train_logger = xlogger('models/history/' + model_name + '_train.dat', append=append)
val_logger = xlogger('models/history/' + model_name + '_val.dat', append=append)
test_logger = xlogger('models/history/' + model_name + '_test.dat', append=append)


In [ ]:
def plot_images(images, save=False, name='images', cmap='viridis', return_axes=False, show=True):
    fig, axes = plt.subplots(2, len(images)//2, figsize=(len(images)//2*0.99, 2))
    fig.subplots_adjust(hspace=0., wspace=0.)
    axes = axes.flatten()
    for ax, img in zip(axes, images):
        ax.imshow(img.permute(1, 2, 0), cmap=cmap)
        ax.axis('off')
    if save:
        direc = 'models/history/'+name+'/'
        os.makedirs(direc, exist_ok=True)
        # check next number
        num = 0
        while glob.glob(f'{direc}{name}_{num}.png'):
            num += 1
        plt.savefig(f'{direc}{name}_{num}.png')
    if show:
        plt.show()
    if return_axes:
        return axes

def nxcorr(outputs, labels):
    dim = int(outputs.shape[-1])
    outputs = outputs.reshape(-1, dim**2)
    labels = labels.reshape(-1, dim**2)
    
    outputs_norm = (outputs.reshape(-1, dim, dim) - torch.nanmean(outputs, axis=1).reshape(-1, 1, 1)) / torch.std(outputs, axis=1).reshape(-1, 1, 1)
    labels_norm = (labels.reshape(-1, dim, dim) - torch.nanmean(labels, axis=1).reshape(-1, 1, 1)) / torch.std(labels, axis=1).reshape(-1, 1, 1)

    fft_outputs = torch.fft.fftn(outputs_norm, s=[outputs_norm.size(d)*1 for d in [1,2]], dim=[1,2])
    fft_labels = torch.fft.fftn(labels_norm, s=[outputs_norm.size(d)*1 for d in [1,2]], dim=[1,2])

    xcorr = torch.fft.ifftn(fft_outputs * torch.conj(fft_labels), dim=[1,2])

    nxcorr_flat = xcorr.reshape(-1, dim**2)
    idx = torch.argmax(torch.abs(nxcorr_flat), dim=1)

    return idx, torch.abs(nxcorr_flat[torch.arange(nxcorr_flat.shape[0]), idx])/dim**2

def shift_image(im1, im2): # shift single im2 by idx
    idx, _ = nxcorr(im1, im2)
    im2 = torch.roll(im2, shifts=int(idx))
    return im1, im2

def shift_all(truth, imgs):
    shifted_imgs = []
    for img in imgs:
        _, shifted_img = shift_image(truth, img)
        shifted_imgs.append(shifted_img)
    return np.array(shifted_imgs)

In [ ]:
# Load initial data

combine_ci_vis = False


ehtim=True
tint_sec = 5   
tadv_sec = 600
tstart_hr = 0
tstop_hr = 24
# psize = 7.757018897750619e-12 * 2
psize = 1.7044214966184275e-11
bw_hz = [230E9]#, 345E9]

batch_size = 64

data_dir = 'data/datasets/'
train_data = ds.ImgDataset(['data/datasets/val/val_mring.npy'], transform=tforms.train_transforms,
                           tint_sec=tint_sec, tadv_sec=tadv_sec, tstart_hr=tstart_hr, tstop_hr=tstop_hr, bw_hz=bw_hz, psize=psize,
                           )
train = DataLoader(train_data, batch_size=batch_size, shuffle=True)

dataiter = iter(train)
static_val = next(dataiter)

clObj = train.dataset.closure

ci_shape = clObj.FTCI(static_val[0]).shape[-1]
data_dim = clObj.FTCI(static_val[0], return_combined=combine_ci_vis).shape[-1]


In [ ]:
# testing with image
import seaborn as sns

cmap = 'Greys'

def gauss_noise(img, sigma):
    out = img + sigma * torch.randn_like(img).numpy()
    return out
   

imgdim = 64

model = torch.load('models/saved_models/'+model_name+'.pt').module.to(device)
model.eval()


load_fits = 'Images/s_gauss.fits'
img = eh.image.load_fits(load_fits)
name = load_fits.split('/')[-1].split('.')[0]
img = img.regrid_image(img.fovx(), imgdim)
im = img.imarr()

integrated_flux = 100

im = im/np.max(im) # normalise the intensity
im = im/np.sum(im)*integrated_flux
im = torch.tensor(im).to(torch.float32).reshape(1, imgdim, imgdim)

invs = torch.tensor(clObj.FTCI(im, add_th_noise=False, return_combined=combine_ci_vis, th_noise_factor=1).reshape(1, -1), dtype=torch.float32).to(device)

out_features, outputs, attns = model.predict_with_attn(invs)
outputs = outputs.cpu().detach().numpy().reshape(imgdim,imgdim)
outputs = (outputs - np.min(outputs))/(np.max(outputs) - np.min(outputs))

im, outputs = im.reshape(imgdim, imgdim), outputs.reshape(imgdim, imgdim)#.numpy()


encoder_latent, _, encoder_out = model.encoder_to_img(torch.Tensor(im).view(1,1,imgdim, imgdim).to(device))
encoder_out = encoder_out.cpu().detach().numpy().reshape(imgdim, imgdim)


fig, ax = plt.subplots(1, 4, figsize=(15,5))
ax[0].imshow(im, cmap=cmap)
ax[0].set_title('Truth')

ax[1].imshow(encoder_out, cmap=cmap)
ax[1].set_title('Encoder Output')
idx, xcorr = nxcorr(torch.tensor(encoder_out), torch.tensor(im))
xcorr = xcorr.cpu().detach().numpy()[0]
ax[1].text(0.5, 0.1, f'{xcorr:.3f}', ha='center', va='center', transform=ax[1].transAxes, color='k')

ax[2].imshow(outputs, cmap=cmap)
ax[2].set_title('CI Output')
idx, xcorr = nxcorr(torch.tensor(outputs), torch.tensor(im))
xcorr = xcorr.cpu().detach().numpy()[0]
ax[2].text(0.5, 0.1, f'{xcorr:.3f}', ha='center', va='center', transform=ax[2].transAxes, color='k')

ax[3].imshow(im-outputs)
ax[3].set_title('CI Residual')

